In [1]:
import os
os.chdir("../")

In [2]:
%pwd

'c:\\Users\\HP\\Desktop\\swatisingh\\ml\\mlops_project\\kidney-disease-classification-project'

In [ ]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class TrainingConfig:
    root_dir: Path
    trained_model_dir: Path
    updated_base_model_dir: Path
    training_data: Path
    params_augmentation: bool
    params_batch_size: int
    params_epochs: int
    params_image_size: list

    

In [ ]:
from kidney_disease_classifier.constants import *
from kidney_disease_classifier.utils.common import read_yaml, create_directories

In [ ]:
from kidney_disease_classifier.components import prepare_base_model
import os

class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])
        

def get_training_config(self) -> TrainingConfig:

    training = self.config.training
    prepare_base_model_config = self.config.prepare_base_model
    params = self.params

    training_data = self.config.data_ingestion.unzip_dir

    create_directories([training.root_dir])

    training_config = TrainingConfig(
        root_dir=Path(training.root_dir),
        trained_model_path=Path(training.trained_model_path),
        updated_base_model_path=Path(prepare_base_model_config.updated_base_model_path),
        training_data=Path(training_data),
        params_augmentation=params.AUGMENTATION,
        params_batch_size=params.BATCH_SIZE,
        params_epochs=params.EPOCHS,
        params_image_size=params.IMAGE_SIZE
    )

    return training_config

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models

# ==============================
# DEVICE
# ==============================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ==============================
# LOAD UPDATED BASE MODEL
# ==============================
model = models.vgg16()

# Replace classifier according to number of classes
model.classifier[6] = nn.Linear(
    4096,
    training_config.params_classes
)

# Load pretrained weights safely
model.load_state_dict(torch.load(
    training_config.updated_base_model_path,
    map_location=device
))

model = model.to(device)

# ==============================
# DATA TRANSFORMS (WITH NORMALIZATION ✅)
# ==============================
transform = transforms.Compose([
    transforms.Resize(tuple(training_config.params_image_size)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ==============================
# LOAD DATASET
# ==============================
full_dataset = datasets.ImageFolder(
    root=training_config.training_data,
    transform=transform
)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_dataset,
    [train_size, val_size]
)

train_loader = DataLoader(
    train_dataset,
    batch_size=training_config.params_batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=training_config.params_batch_size,
    shuffle=False
)

# ==============================
# LOSS + OPTIMIZER
# ==============================
criterion = nn.CrossEntropyLoss()

# Train only last layer (transfer learning ✅)
optimizer = torch.optim.Adam(
    model.classifier[6].parameters(),
    lr=training_config.params_learning_rate
)

epochs = training_config.params_epochs

# ==============================
# TRAINING LOOP
# ==============================
for epoch in range(epochs):

    # -------- TRAIN --------
    model.train()
    train_loss = 0
    correct = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()

    train_loss /= len(train_loader)
    train_accuracy = 100 * correct / len(train_dataset)

    # -------- VALIDATION --------
    model.eval()
    val_loss = 0
    val_correct = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            val_correct += (predicted == labels).sum().item()

    val_loss /= len(val_loader)
    val_accuracy = 100 * val_correct / len(val_dataset)

    print(f"Epoch [{epoch+1}/{epochs}] "
          f"Train Loss: {train_loss:.4f} | "
          f"Train Acc: {train_accuracy:.2f}% | "
          f"Val Loss: {val_loss:.4f} | "
          f"Val Acc: {val_accuracy:.2f}%")

# ==============================
# SAVE TRAINED MODEL
# ==============================
torch.save(
    model.state_dict(),
    training_config.trained_model_path
)

print("✅ Training Complete — Model Saved Successfully")

In [ ]:
try:
    config_manager = ConfigurationManager()
    training_config = config_manager.get_training_config()

    model_trainer = ModelTrainer(config=training_config)

    model_trainer.train()

except Exception as e:
    raise e